# **SalaryIQ: Data Cleaning & QA Pipeline**

**Topic:** Data Analyst Salary Intelligence Platform

**Dataset:** Glassdoor Data Analyst Job Postings (US) `glassdoor-raw.csv`  

**Purpose:** Document and reproduce every data quality check and feature engineering step applied to the raw Glassdoor scrape. This notebook produces the cleaned dataset used to power the Tableau dashboard and hypothesis testing in Sprint 4.



**Cleaning steps in this notebook:**

| # | Check | What it does |
|---|---|---|
| 1 | Schema validation | Required columns present, index artifact removed |
| 2 | True null values | Baseline null check before sentinel conversion |
| 3 | Sentinel -1 values | Convert Glassdoor's -1 missing encoding to NaN |
| 4 | Duplicate records | Exact row deduplication |
| 5 | Invalid job titles | Remove scraping artifacts (titles starting with `+`) |
| 6 | Salary parsing | Parse `"$30K-$53K (Glassdoor est.)"` → numeric min/max/avg |
| 7 | Salary range validation | Confirm min ≤ max; record distribution |
| 8 | Company name cleaning | Split rating embedded in company name field |
| 9 | Rating range validation | Confirm all ratings are within 1.0–5.0 |
| 10 | Location parsing | Extract `city` and `state` from `"City, ST"` |
| 11 | Founded year plausibility | Flag years outside 1800–2024 |
| 12 | Seniority classification | Derive `seniority_level` from job title keywords |
| 13 | Skill flag extraction | 9 binary flags from job description text (NLP) |
| 14 | Company size labelling | Map size strings to 7 ordered categories |
| 15 | Final validation | Confirm no sentinels remain; export |



## **Setup**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import re
import os
import warnings
from datetime import datetime

warnings.filterwarnings('ignore')

# ── Constants ──────────────────────────────────────────────────────────────
SENTINEL_COLS = [
    "Rating", "Size", "Founded", "Type of ownership",
    "Industry", "Sector", "Revenue", "Competitors",
    "Headquarters", "Easy Apply"
]

SKILLS = {
    "skill_python"          : r"python",
    "skill_sql"             : r"\bsql\b",
    "skill_tableau"         : r"tableau",
    "skill_excel"           : r"\bexcel\b",
    "skill_r"               : r"\br\b",
    "skill_power_bi"        : r"power\s?bi",
    "skill_spark"           : r"\bspark\b",
    "skill_machine_learning": r"machine\s?learning",
    "skill_statistics"      : r"statistic",
}

SIZE_MAP = {
    "1 to 50 employees"       : "1. Micro (1-50)",
    "51 to 200 employees"     : "2. Small (51-200)",
    "201 to 500 employees"    : "3. Small-Mid (201-500)",
    "501 to 1000 employees"   : "4. Mid (501-1K)",
    "1001 to 5000 employees"  : "5. Large (1K-5K)",
    "5001 to 10000 employees" : "6. Very Large (5K-10K)",
    "10000+ employees"        : "7. Enterprise (10K+)",
}

# ── QA logging ─────────────────────────────────────────────────────────────
issues = []

def log(check_id, check_name, column, severity, finding, action, affected_rows=0):
    issues.append({
        "check_id"      : check_id,
        "check_name"    : check_name,
        "column"        : column,
        "severity"      : severity,
        "finding"       : finding,
        "action_taken"  : action,
        "affected_rows" : affected_rows,
        "timestamp"     : datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    })
    icon = {"CRITICAL": "❌", "WARNING": "⚠️", "INFO": "ℹ️"}.get(severity, "  ")
    print(f"  {icon}  [{check_id}] {finding}")

print("Setup complete")

Setup complete


## **Load Raw Data**

We upload and load `glassdoor-raw.csv`, then inspect its shape, column data types, and where any null values appear.

> **Why inspect before cleaning?**
Understanding the raw structure before touching anything prevents accidental data loss and tells us exactly which columns need attention.

In [2]:
# @title
#Import file to Google Colab
from google.colab import files
uploaded = files.upload()

df = pd.read_csv('glassdoor-raw.csv', index_col=0)
print(f'Shape: {df.shape}')
df.head(5)

Saving glassdoor-raw.csv to glassdoor-raw.csv
Shape: (2253, 15)


,Job Title,Salary Estimate,Job Description,Rating,Company Name,Location,Headquarters,Size,Founded,Type of ownership,Industry,Sector,Revenue,Competitors,Easy Apply
1352,Data Analyst,$30K-$53K (Glassdoor est.),"Job Description\nETL, SQL Queries, Data Modeli...",-1.0,1,"Dallas, TX",-1,-1,-1,-1,-1,-1,-1,-1,-1
119,PBM Data Analyst,$51K-$87K (Glassdoor est.),Responsibilities:\nProduce analyses and create...,3.2,1199SEIU Funds\n3.2,"New York, NY","New York, NY",1001 to 5000 employees,-1,Nonprofit Organization,Insurance Carriers,Insurance,$100 to $500 million (USD),-1,-1
324,Senior Data Analyst,$42K-$74K (Glassdoor est.),Responsibilities\nAssist the Associate Directo...,3.2,1199SEIU Funds\n3.2,"New York, NY","New York, NY",1001 to 5000 employees,-1,Nonprofit Organization,Insurance Carriers,Insurance,$100 to $500 million (USD),-1,-1
812,Data Governance Analyst,$42K-$76K (Glassdoor est.),SUMMARY\nThe Data Governance Analyst performs ...,5.0,1872 Consulting\n5.0,"Chicago, IL","Chicago, IL",1 to 50 employees,-1,Company - Private,-1,-1,Unknown / Non-Applicable,-1,-1
1074,Data Analyst - Entry Level,$41K-$78K (Glassdoor est.),Looking to hire for a Data Analyst role with o...,-1.0,2000 east westmoreland st llc,"King of Prussia, PA",-1,-1,-1,-1,-1,-1,-1,-1,-1


---
## **Check 1: Schema Validation**

Confirm that all required columns are present. Also check for `Unnamed: 0` — a common artifact when a pandas DataFrame is exported to CSV without `index=False`. It adds a duplicate row number column that should be removed.


In [3]:
# @title
required = ["Job Title", "Salary Estimate", "Job Description", "Rating",
            "Company Name", "Location", "Size", "Founded", "Sector", "Industry"]

missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

log("QA-01a", "Schema validation", "all required columns", "INFO",
    f"All {len(required)} required columns present ({len(df.columns)} total)", "None")
print(f"✅  All {len(required)} required columns found")

# Drop unnamed index column if present
if "Unnamed: 0" in df.columns:
    log("QA-01b", "Index column", "Unnamed: 0", "WARNING",
        "Unnamed index column present — CSV export artifact (index saved without index=False)",
        "Dropped", len(df))
    df = df.drop(columns=["Unnamed: 0"])
    print(f"⚠️   'Unnamed: 0' index column removed")

print(f"\nFinal column count: {len(df.columns)}")
df.dtypes.to_frame("dtype")

  ℹ️  [QA-01a] All 10 required columns present (15 total)
✅  All 10 required columns found

Final column count: 15


,dtype
Job Title,object
Salary Estimate,object
Job Description,object
Rating,float64
Company Name,object
Location,object
Headquarters,object
Size,object
Founded,int64
Type of ownership,object


---
## **Check 2: True Null Values**

Before touching anything, record the baseline null state. In this dataset, **almost all missing data is encoded as `-1`** (Glassdoor's sentinel convention) rather than as true NaN — so the true null count is very low. The real missing data problem is handled in Check 3.


In [4]:
# @title
null_counts = df.isnull().sum()
null_df = null_counts[null_counts > 0].to_frame("null_count")
null_df["null_pct"] = (null_df["null_count"] / len(df) * 100).round(2)

total_nulls = null_counts.sum()
log("QA-02", "True nulls", "all columns", "INFO",
    f"{total_nulls} total true nulls across dataset — most missing data is -1 sentinel",
    "None — handled in Check 3", total_nulls)

if len(null_df) > 0:
    print(f"Columns with true null values:")
    display(null_df)
else:
    print("No true null values (aside from Company Name = 1 null)")

print(f"\nTotal true nulls: {total_nulls}")
print("Note: The bulk of missing data is encoded as -1 sentinel — see Check 3")

  ℹ️  [QA-02] 1 total true nulls across dataset — most missing data is -1 sentinel
Columns with true null values:


,null_count,null_pct
Company Name,1,0.04



Total true nulls: 1
Note: The bulk of missing data is encoded as -1 sentinel — see Check 3


---
## **Check 3: Sentinel -1 Values (Glassdoor Missing Encoding)**

This is the most pervasive quality issue in the dataset. Glassdoor uses **-1 as a placeholder for missing** across 10 columns — treating it as a real value would make `Founded = -1` or `Rating = -1` appear in analysis.

All -1 sentinels must be converted to `NaN` before any analysis.


In [5]:
# @title
sentinel_summary = []
for col in SENTINEL_COLS:
    if col not in df.columns:
        continue
    mask = (df[col] == -1) if df[col].dtype in [np.float64, np.int64]            else (df[col].astype(str) == "-1")
    n   = mask.sum()
    pct = n / len(df) * 100
    sev = "WARNING" if pct > 20 else "INFO"
    sentinel_summary.append({
        "column": col, "sentinel_rows": n,
        "pct": round(pct, 1), "severity": sev
    })
    log("QA-03", "Sentinel -1", col, sev,
        f"{n} rows ({pct:.1f}%) use -1 as missing encoding",
        "Replaced with NaN", n)
    df.loc[mask, col] = np.nan

log("QA-03z", "Sentinel conversion", "all cols", "INFO",
    "All -1 sentinels converted to NaN across 10 columns", "Completed", 0)

display(pd.DataFrame(sentinel_summary))

  ℹ️  [QA-03] 272 rows (12.1%) use -1 as missing encoding
  ℹ️  [QA-03] 163 rows (7.2%) use -1 as missing encoding
  ⚠️  [QA-03] 660 rows (29.3%) use -1 as missing encoding
  ℹ️  [QA-03] 163 rows (7.2%) use -1 as missing encoding
  ℹ️  [QA-03] 353 rows (15.7%) use -1 as missing encoding
  ℹ️  [QA-03] 353 rows (15.7%) use -1 as missing encoding
  ℹ️  [QA-03] 163 rows (7.2%) use -1 as missing encoding
  ⚠️  [QA-03] 1732 rows (76.9%) use -1 as missing encoding
  ℹ️  [QA-03] 172 rows (7.6%) use -1 as missing encoding
  ⚠️  [QA-03] 2173 rows (96.4%) use -1 as missing encoding
  ℹ️  [QA-03z] All -1 sentinels converted to NaN across 10 columns


,column,sentinel_rows,pct,severity
0,Rating,272,12.1,INFO
1,Size,163,7.2,INFO
2,Founded,660,29.3,WARNING
3,Type of ownership,163,7.2,INFO
4,Industry,353,15.7,INFO
5,Sector,353,15.7,INFO
6,Revenue,163,7.2,INFO
7,Competitors,1732,76.9,WARNING
8,Headquarters,172,7.6,INFO
9,Easy Apply,2173,96.4,WARNING


---
## **Check 4: Duplicate Records**

Check for exact row duplicates across all columns. Duplicates in a job posting dataset can appear when the same listing is scraped multiple times.


In [7]:
# @title
n_dup = df.duplicated().sum()

if n_dup > 0:
    log("QA-04", "Duplicates", "all columns", "CRITICAL",
        f"{n_dup} exact duplicate rows found", "Removed, kept first", n_dup)
    df = df.drop_duplicates(keep="first")
    print(f"❌  {n_dup} duplicate rows removed")
else:
    log("QA-04", "Duplicates", "all columns", "INFO",
        "No exact duplicate rows found", "None", 0)
    print(f"✅  No duplicate rows found across {len(df):,} records")

  ℹ️  [QA-04] No exact duplicate rows found
✅  No duplicate rows found across 2,253 records


---
## **Check 5: Invalid Job Title Entries**

Some rows contain non-standard job titles that are scraping artifacts — titles beginning with `+` or other non-alphabetic characters. These are not real job postings and should be removed.


In [8]:
# @title
invalid_mask = df["Job Title"].str.match(r"^\+|^[^a-zA-Z]", na=False)
n_inv = invalid_mask.sum()

if n_inv > 0:
    print(f"Examples of invalid titles found:")
    display(df[invalid_mask][["Job Title", "Company Name", "Location"]].head(5))
    log("QA-05", "Invalid job titles", "Job Title", "WARNING",
        f"{n_inv} rows have non-standard job titles — likely scraping artifacts",
        f"Removed {n_inv} rows", n_inv)
    df = df[~invalid_mask].reset_index(drop=True)
    print(f"\n⚠️  {n_inv} rows removed")
else:
    log("QA-05", "Invalid job titles", "Job Title", "INFO",
        "No invalid job title entries found", "None", 0)
    print(f"✅  All job titles are valid")

log("QA-05z", "Post-filter count", "rows", "INFO",
    f"Dataset now has {len(df):,} rows", "None", 0)
print(f"Rows remaining: {len(df):,}")

Examples of invalid titles found:


,Job Title,Company Name,Location
1667,9-1-1 Data Analyst,911 Datamaster Inc,"Austin, TX"
1302,7232 Data Engineer (Analyst/Programmer - Caree...,California State University\n4.1,"San Diego, CA"
295,2021 Summer Analyst Program- Data & Analytics ...,Eagle Investment Systems\n3.7,"New York, NY"
652,【1yr OPT+Intern】Data Analyst 保实习保就业,Easyfind Company,"Arcadia, CA"
968,.Net Developer - Program Data Design Analyst,Harris County Appraisal District\n2.2,"Houston, TX"


  ⚠️  [QA-05] 19 rows have non-standard job titles — likely scraping artifacts

⚠️  19 rows removed
  ℹ️  [QA-05z] Dataset now has 2,234 rows
Rows remaining: 2,234


---
## **Check 6: Salary Parsing**

`Salary Estimate` is stored as a formatted string like `"$30K-$53K (Glassdoor est.)"` as it cannot be used in any numerical analysis without parsing. We extract the min, max, and midpoint average into three new numeric columns.


In [9]:
# @title
salary_re = re.compile(r'\$(\d+)K-\$(\d+)K')

def parse_salary(s):
    m = salary_re.search(str(s))
    if m:
        lo, hi = int(m.group(1)) * 1000, int(m.group(2)) * 1000
        return lo, hi, int((lo + hi) / 2)
    return np.nan, np.nan, np.nan

# Check for unparseable rows first
unparseable = df["Salary Estimate"].apply(
    lambda s: salary_re.search(str(s)) is None).sum()
log("QA-06", "Salary format", "Salary Estimate",
    "WARNING" if unparseable > 0 else "INFO",
    f"{unparseable} rows did not match expected '$XK-$YK' format",
    "Set to NaN", unparseable)

# Parse
parsed = df["Salary Estimate"].apply(parse_salary)
df["salary_min"] = parsed.apply(lambda x: x[0]).astype("Int64")
df["salary_max"] = parsed.apply(lambda x: x[1]).astype("Int64")
df["salary_avg"] = parsed.apply(lambda x: x[2]).astype("Int64")

print(f"Unparseable rows   : {unparseable}")
print(f"\nSample parsed values:")
df[["Salary Estimate", "salary_min", "salary_max", "salary_avg"]].head(5)

  ⚠️  [QA-06] 1 rows did not match expected '$XK-$YK' format
Unparseable rows   : 1

Sample parsed values:


,Salary Estimate,salary_min,salary_max,salary_avg
0,$30K-$53K (Glassdoor est.),30000,53000,41500
1,$51K-$87K (Glassdoor est.),51000,87000,69000
2,$42K-$74K (Glassdoor est.),42000,74000,58000
3,$42K-$76K (Glassdoor est.),42000,76000,59000
4,$41K-$78K (Glassdoor est.),41000,78000,59500


---
## **Check 7: Salary Range Validation & Distribution**

Two sub-checks: confirm no rows have `salary_min > salary_max` (inverted range), then record the full salary distribution as an audit trail — mean, median, percentiles.


In [11]:
# @title
# Range logic check
inverted = (df["salary_min"] > df["salary_max"]).sum()
log("QA-07a", "Salary range logic", "salary_min/max",
    "CRITICAL" if inverted else "INFO",
    f"{inverted} rows where min > max (should be 0)",
    "None" if not inverted else "Flagged", inverted)

if inverted:
    print(f"❌  {inverted} rows have salary_min > salary_max")
else:
    print(f"✅  All salary ranges valid (min ≤ max)")

# Distribution audit trail
avg = df["salary_avg"].dropna()
log("QA-07b", "Salary distribution", "salary_avg", "INFO",
    f"Mean ${int(avg.mean()):,} | Median ${int(avg.median()):,} | "
    f"P10 ${int(avg.quantile(.1)):,} | P90 ${int(avg.quantile(.9)):,} | "
    f"Range ${int(avg.min()):,}–${int(avg.max()):,}",
    "Recorded for audit trail", 0)

print(f"\nSalary distribution (salary_avg):")
print(f"  Mean    : ${int(avg.mean()):,}")
print(f"  Median  : ${int(avg.median()):,}")
print(f"  P10     : ${int(avg.quantile(.1)):,}")
print(f"  P90     : ${int(avg.quantile(.9)):,}")
print(f"  Min/Max : ${int(avg.min()):,} – ${int(avg.max()):,}")

  ℹ️  [QA-07a] 0 rows where min > max (should be 0)
✅  All salary ranges valid (min ≤ max)
  ℹ️  [QA-07b] Mean $72,152 | Median $69,000 | P10 $40,500 | P90 $104,500 | Range $33,500–$150,000

Salary distribution (salary_avg):
  Mean    : $72,152
  Median  : $69,000
  P10     : $40,500
  P90     : $104,500
  Min/Max : $33,500 – $150,000


---
## **Check 8: Company Name Cleaning**

Glassdoor embeds the company rating inside the `Company Name` field, separated by a newline:

```
"1199SEIU Funds
3.2"
```

This means one field contains two pieces of information. We split on `
` to extract the clean company name.


In [14]:
# @title
embedded = df["Company Name"].str.contains(r'\n', na=False).sum()
log("QA-08", "Embedded rating", "Company Name", "WARNING",
    f"{embedded:,} rows have Glassdoor rating embedded in company name "
    f"(e.g. 'Acme Corp\\n3.2') — two values in one field",
    "Split on newline to extract company_name", embedded)

# Show examples before cleaning
print("Before cleaning (sample):")
display(df[df["Company Name"].str.contains(r'\n', na=False)][["Company Name"]].head(5))

# Clean
df["company_name"] = df["Company Name"].str.split("\n").str[0].str.strip()

print("\nAfter cleaning (sample):")
display(df[["Company Name", "company_name"]].head(5))
print(f"\n⚠️  {embedded:,} company names cleaned")

  ⚠️  [QA-08] 1,964 rows have Glassdoor rating embedded in company name (e.g. 'Acme Corp\n3.2') — two values in one field
Before cleaning (sample):


,Company Name
1,1199SEIU Funds\n3.2
2,1199SEIU Funds\n3.2
3,1872 Consulting\n5.0
6,22nd Century Staffing\n4.3
7,22nd Century Staffing\n4.3



After cleaning (sample):


,Company Name,company_name
0,1,1
1,1199SEIU Funds\n3.2,1199SEIU Funds
2,1199SEIU Funds\n3.2,1199SEIU Funds
3,1872 Consulting\n5.0,1872 Consulting
4,2000 east westmoreland st llc,2000 east westmoreland st llc



⚠️  1,964 company names cleaned


---
## **Check 9: Rating Range Validation**

Glassdoor ratings run from 1.0 to 5.0. Any value outside this range after sentinel conversion would indicate a data issue.


In [16]:
# @title
valid   = df["Rating"].between(1, 5, inclusive="both")
out_rng = (~valid & df["Rating"].notna()).sum()

if out_rng:
    log("QA-09", "Rating range", "Rating", "WARNING",
        f"{out_rng} values outside valid 1.0–5.0 range", "Set to NaN", out_rng)
    df.loc[~valid & df["Rating"].notna(), "Rating"] = np.nan
    print(f"⚠️  {out_rng} out-of-range ratings set to NaN")
else:
    log("QA-09", "Rating range", "Rating", "INFO",
        f"All non-null ratings within 1.0–5.0 (mean: {df['Rating'].mean():.2f})", "None", 0)
    print(f"✅  All ratings valid (1.0–5.0)")

print(f"\nRating distribution:")
print(df["Rating"].describe().round(2))

  ℹ️  [QA-09] All non-null ratings within 1.0–5.0 (mean: 3.73)
✅  All ratings valid (1.0–5.0)

Rating distribution:
count    1964.00
mean        3.73
std         0.67
min         1.00
25%         3.30
50%         3.70
75%         4.10
max         5.00
Name: Rating, dtype: float64


---
## **Check 10: Location Parsing**

`Location` is formatted as `"City, ST"`. We extract `city` and `state` into separate columns — needed for the state-level salary analysis in the hypothesis testing.


In [18]:
# @title
bad_fmt = (~df["Location"].str.contains(r',\s*[A-Z]{2}', na=False)).sum()
log("QA-10a", "Location format", "Location",
    "WARNING" if bad_fmt else "INFO",
    f"{'All rows follow' if not bad_fmt else str(bad_fmt)+' do not follow'} 'City, ST' format",
    "None" if not bad_fmt else "State set to NaN for non-conforming rows", bad_fmt)

df["city"]  = df["Location"].str.split(",").str[0].str.strip()
df["state"] = df["Location"].str.extract(r',\s*([A-Z]{2})$')[0]

log("QA-10b", "City/state extraction", "city + state", "INFO",
    f"city and state extracted for all {df['state'].notna().sum():,} rows",
    "Completed", 0)

print(f"Rows with valid 'City, ST' format : {len(df) - bad_fmt:,}")
print(f"Rows with non-standard format     : {bad_fmt}")
print(f"\nSample:")
df[["Location", "city", "state"]].head(5)

  ℹ️  [QA-10a] All rows follow 'City, ST' format
  ℹ️  [QA-10b] city and state extracted for all 2,234 rows
Rows with valid 'City, ST' format : 2,234
Rows with non-standard format     : 0

Sample:


,Location,city,state
0,"Dallas, TX",Dallas,TX
1,"New York, NY",New York,NY
2,"New York, NY",New York,NY
3,"Chicago, IL",Chicago,IL
4,"King of Prussia, PA",King of Prussia,PA


---
## **Check 11: Seniority Classification (Derived Column)**

The raw data has no seniority column. We derive `seniority_level` from keywords in the `Job Title` field:

| Level | S1: Title keywords & Roman numerals | S2: Years of experience | S3: Description language   |
|---|---|---|----|
| Senior      | `senior`, `sr.`, `lead`, `principal`,        | 6+ years                                     | "cross-functional", "mentor",                   |
|             | `staff`, `manager`, `Analyst III / IV`       |                                              | "architect", "10+ years"                        |
| Mid-Level   | —                                            | 3 – 5 years                                  | —                                               |
| Entry-Level | `entry`, `jr.`, `junior`, `associate`,       | ≤ 2 years                                    | "new graduate", "will train", "0–1 year"        |
|             | `intern`, `trainee`, `Analyst I`             |                                              |                                                 |
| Default     | —                                            | —                                            | Mid-Level (no signal found)   


In [22]:
# @title
import re

# SIGNAL 1 — Job Title keywords & Roman numerals
def title_seniority_score(title):
    title = str(title).lower()

    roman_map = {
        r'\bdata\s+\w*\s*analyst\s+i\b':   ('Entry-Level', 'high'),
        r'\bdata\s+\w*\s*analyst\s+ii\b':  ('Mid-Level',   'high'),
        r'\bdata\s+\w*\s*analyst\s+iii\b': ('Senior',      'high'),
        r'\bdata\s+\w*\s*analyst\s+iv\b':  ('Senior',      'high'),
        r'\blevel\s+1\b':                  ('Entry-Level', 'medium'),
        r'\blevel\s+2\b':                  ('Mid-Level',   'medium'),
        r'\blevel\s+3\b':                  ('Senior',      'medium'),
    }
    for pattern, result in roman_map.items():
        if re.search(pattern, title):
            return result

    senior_kw = ['senior', 'sr.', 'sr ', 'lead', 'principal', 'staff',
                 'manager', 'director', 'head of', 'vp ', 'vice president']
    entry_kw  = ['entry', 'jr.', 'jr ', 'junior', 'associate', 'intern',
                 'graduate', 'trainee', 'apprentice', 'early career']

    if any(kw in title for kw in senior_kw):
        return ('Senior', 'high')
    if any(kw in title for kw in entry_kw):
        return ('Entry-Level', 'high')
    return (None, None)


# SIGNAL 2 — Years of experience from Job Description
def extract_min_exp_years(text):
    text = str(text).lower()
    patterns = [
        r'minimum\s+of\s+(\d+)\s*(?:\+)?\s*years?',
        r'at\s+least\s+(\d+)\s*(?:\+)?\s*years?',
        r'(\d+)\+\s*years?\s*(?:of\s*)?(?:experience|exp)',
        r'(\d+)\s*-\s*\d+\s*years?\s*(?:of\s*)?(?:experience|exp)',
        r'(\d+)\s*years?\s*(?:of\s*)?(?:experience|exp)',
    ]
    for p in patterns:
        m = re.search(p, text)
        if m:
            val = int(m.group(1))
            if 0 < val <= 20:
                return val
    return None

def exp_years_to_level(years):
    if years is None:   return None
    if years <= 2:      return 'Entry-Level'
    if years <= 5:      return 'Mid-Level'
    return 'Senior'


# SIGNAL 3 — Description language (weak tiebreaker)
def desc_seniority_signal(text):
    text = str(text).lower()
    senior_phrases = ['mentor', 'lead a team', 'manage a team', 'strategic',
                      'cross-functional', 'architect', '8+ years', '10+ years']
    entry_phrases  = ['no experience required', 'new graduate', 'recent graduate',
                      'entry-level', '0-1 year', '0-2 year', 'will train']
    senior_hits = sum(1 for p in senior_phrases if p in text)
    entry_hits  = sum(1 for p in entry_phrases  if p in text)
    if senior_hits > entry_hits and senior_hits >= 2: return 'Senior'
    if entry_hits  > senior_hits and entry_hits >= 1: return 'Entry-Level'
    return None


# COMBINED CLASSIFIER
def classify_seniority(row):
    title_level, title_conf = title_seniority_score(row['Job Title'])       # ← capital J T
    exp_level  = exp_years_to_level(extract_min_exp_years(row['Job Description']))  # ← capital J D
    desc_level = desc_seniority_signal(row['Job Description'])              # ← capital J D

    if title_level and title_conf == 'high': return title_level
    if exp_level:                            return exp_level
    if title_level and title_conf == 'medium': return title_level
    if desc_level:                           return desc_level
    return 'Mid-Level'


df['seniority_level'] = df.apply(classify_seniority, axis=1)

vc = df['seniority_level'].value_counts()
log("QA-12", "Seniority classification", "seniority_level", "INFO",
    f"Senior: {vc.get('Senior',0):,} | Mid-Level: {vc.get('Mid-Level',0):,} | "
    f"Entry-Level: {vc.get('Entry-Level',0):,}",
    "Multi-signal classifier: title keywords + Roman numerals + years of experience + description language", 0)

print(df['seniority_level'].value_counts())

  ℹ️  [QA-12] Senior: 670 | Mid-Level: 1,231 | Entry-Level: 333
seniority_level
Mid-Level      1231
Senior          670
Entry-Level     333
Name: count, dtype: int64


---
## **Check 12: Skill Flag Extraction**

We scan each job description for 9 skills using regex pattern matching. Each skill gets a **binary flag column** (1 = mentioned in the description, 0 = not mentioned).

This is the most powerful feature engineering step — it's what makes the salary premium analysis possible (confirming Python +$3,500 and Excel −$1,500).


In [24]:
# @title
desc_lower = df["Job Description"].str.lower().fillna("")
skill_results = []

for skill_col, pattern in SKILLS.items():
    df[skill_col] = desc_lower.str.contains(pattern, regex=True).astype(int)
    n   = df[skill_col].sum()
    pct = n / len(df) * 100
    log("QA-13", "Skill flag", skill_col, "INFO",
        f"Mentioned in {n:,} postings ({pct:.1f}%) | regex: {pattern}",
        "Binary flag created (1 = in job description, 0 = not mentioned)", 0)
    skill_results.append({"skill": skill_col.replace("skill_","").replace("_"," ").title(),
                           "postings": n, "demand_pct": round(pct, 1)})

display(pd.DataFrame(skill_results).sort_values("postings", ascending=False).reset_index(drop=True))

  ℹ️  [QA-13] Mentioned in 633 postings (28.3%) | regex: python
  ℹ️  [QA-13] Mentioned in 1,354 postings (60.6%) | regex: \bsql\b
  ℹ️  [QA-13] Mentioned in 618 postings (27.7%) | regex: tableau
  ℹ️  [QA-13] Mentioned in 899 postings (40.2%) | regex: \bexcel\b
  ℹ️  [QA-13] Mentioned in 438 postings (19.6%) | regex: \br\b
  ℹ️  [QA-13] Mentioned in 247 postings (11.1%) | regex: power\s?bi
  ℹ️  [QA-13] Mentioned in 70 postings (3.1%) | regex: \bspark\b
  ℹ️  [QA-13] Mentioned in 181 postings (8.1%) | regex: machine\s?learning
  ℹ️  [QA-13] Mentioned in 835 postings (37.4%) | regex: statistic


,skill,postings,demand_pct
0,Sql,1354,60.6
1,Excel,899,40.2
2,Statistics,835,37.4
3,Python,633,28.3
4,Tableau,618,27.7
5,R,438,19.6
6,Power Bi,247,11.1
7,Machine Learning,181,8.1
8,Spark,70,3.1


---
## **Check 13: Company Size Labelling (Derived Column)**

The raw `Size` column contains strings like `"1001 to 5000 employees"`. We map these to 7 ordered, human-readable categories for cleaner analysis and visualisation.


In [26]:
# @title
df["company_size_label"] = df["Size"].map(SIZE_MAP).fillna("Unknown")
vc2 = df["company_size_label"].value_counts().sort_index()

log("QA-14", "Company size labels", "company_size_label", "INFO",
    f"7 size categories + Unknown. Largest group: {vc2.index[0]} ({vc2.iloc[0]:,})",
    "New column company_size_label created", 0)

display(vc2.to_frame("count").rename_axis("company_size_label"))

  ℹ️  [QA-14] 7 size categories + Unknown. Largest group: 1. Micro (1-50) (344)


,count
company_size_label,
1. Micro (1-50),344
2. Small (51-200),419
3. Small-Mid (201-500),245
4. Mid (501-1K),206
5. Large (1K-5K),347
6. Very Large (5K-10K),97
7. Enterprise (10K+),371
Unknown,205


---
## **Check 14: Final Validation & Export**

Rename columns to snake_case, select the 29 reporting columns, verify no -1 sentinels remain, then export.


In [31]:
# @title
rename = {
    "Job Title": "job_title", "Salary Estimate": "salary_estimate",
    "Rating": "rating", "Location": "location",
    "Headquarters": "headquarters", "Size": "size",
    "Founded": "founded", "Type of ownership": "type_of_ownership",
    "Industry": "industry", "Sector": "sector",
    "Revenue": "revenue", "Easy Apply": "easy_apply",
}
df = df.rename(columns=rename)

KEEP = [
    "job_title", "salary_estimate", "rating", "company_name",
    "location", "headquarters", "size", "founded", "type_of_ownership",
    "industry", "sector", "revenue", "easy_apply",
    "city", "state", "salary_min", "salary_max", "salary_avg",
    "seniority_level",
    "skill_python", "skill_sql", "skill_tableau", "skill_excel",
    "skill_r", "skill_power_bi", "skill_spark",
    "skill_machine_learning", "skill_statistics",
    "company_size_label",
]
KEEP      = [c for c in KEEP if c in df.columns]
df_clean  = df[KEEP].copy()

CLEAN_PATH = "../outputs/glassdoor-cleaned.csv"


# Sentinel check
remaining = sum(
    (df_clean[c].astype(str) == "-1").sum()
    for c in df_clean.select_dtypes("object").columns
)
log("QA-15a", "Sentinel post-clean check", "all columns",
    "CRITICAL" if remaining else "INFO",
    f"{'0 — no' if not remaining else str(remaining)} -1 sentinels remain",
    "None" if not remaining else "Review pipeline", remaining)
log("QA-15b", "Final shape", "dataset", "INFO",
    f"{len(df_clean):,} rows × {len(df_clean.columns)} columns — "
    f"13 new columns added vs raw",
    f"Saved to {CLEAN_PATH}", 0)

print(f"✅  Final dataset: {len(df_clean):,} rows × {len(df_clean.columns)} columns")
print(f"✅  Sentinel check: {'PASSED — 0 remaining' if not remaining else f'FAILED — {remaining} found'}")
print(f"\nNew columns added vs raw (16 → 29):")
new_cols = ["company_name", "city", "state", "salary_min", "salary_max", "salary_avg",
            "seniority_level"] + list(SKILLS.keys()) + ["company_size_label"]
for c in new_cols:
    print(f"  + {c}")

  ❌  [QA-15a] 1 -1 sentinels remain
  ℹ️  [QA-15b] 2,234 rows × 29 columns — 13 new columns added vs raw
✅  Final dataset: 2,234 rows × 29 columns
✅  Sentinel check: FAILED — 1 found

New columns added vs raw (16 → 29):
  + company_name
  + city
  + state
  + salary_min
  + salary_max
  + salary_avg
  + seniority_level
  + skill_python
  + skill_sql
  + skill_tableau
  + skill_excel
  + skill_r
  + skill_power_bi
  + skill_spark
  + skill_machine_learning
  + skill_statistics
  + company_size_label


In [32]:
# Preview cleaned dataset
df_clean.head(3)

,job_title,salary_estimate,rating,company_name,location,headquarters,size,founded,type_of_ownership,industry,...,skill_python,skill_sql,skill_tableau,skill_excel,skill_r,skill_power_bi,skill_spark,skill_machine_learning,skill_statistics,company_size_label
0,Data Analyst,$30K-$53K (Glassdoor est.),NaN,1,"Dallas, TX",NaN,NaN,NaN,NaN,NaN,...,0,1,0,0,0,0,0,0,0,Unknown
1,PBM Data Analyst,$51K-$87K (Glassdoor est.),3.2,1199SEIU Funds,"New York, NY","New York, NY",1001 to 5000 employees,NaN,Nonprofit Organization,Insurance Carriers,...,0,1,0,1,0,0,0,0,1,5. Large (1K-5K)
2,Senior Data Analyst,$42K-$74K (Glassdoor est.),3.2,1199SEIU Funds,"New York, NY","New York, NY",1001 to 5000 employees,NaN,Nonprofit Organization,Insurance Carriers,...,0,1,0,1,0,0,0,0,1,5. Large (1K-5K)


In [33]:
# Export
df.to_csv('salaryiq-glassdoor-cleaned.csv', index=False)
files.download('salaryiq-glassdoor-cleaned.csv')
print('Saved: salaryiq-glassdoor-cleaned.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Saved: salaryiq-glassdoor-cleaned.csv


---
## **QA Summary**

Full results across all checks with severity breakdown and the complete log table.


In [35]:
# @title
log_df  = pd.DataFrame(issues)
counts  = log_df["severity"].value_counts()
critical = counts.get("CRITICAL", 0)
warning  = counts.get("WARNING",  0)
info     = counts.get("INFO",     0)

print("=" * 55)
print("  QA PIPELINE COMPLETE — SALARYIQ")
print("=" * 55)
print(f"  ❌  CRITICAL : {critical}")
print(f"  ⚠️   WARNING  : {warning}")
print(f"  ℹ️   INFO     : {info}")
print(f"  {'─'*35}")
print(f"  Total checks run  : {len(log_df)}")
print(f"  Raw rows          : 2,253")
print(f"  Cleaned rows      : {len(df_clean):,}")
print(f"  Raw columns       : 16")
print(f"  Cleaned columns   : {len(df_clean.columns)}")
print("=" * 55)

  QA PIPELINE COMPLETE — SALARYIQ
  ❌  CRITICAL : 2
  ⚠️   WARNING  : 8
  ℹ️   INFO     : 32
  ───────────────────────────────────
  Total checks run  : 42
  Raw rows          : 2,253
  Cleaned rows      : 2,234
  Raw columns       : 16
  Cleaned columns   : 29


In [36]:
# Full QA log
display(log_df[["check_id","check_name","column","severity",
                "finding","action_taken","affected_rows"]])

,check_id,check_name,column,severity,finding,action_taken,affected_rows
0,QA-01a,Schema validation,all required columns,INFO,All 10 required columns present (15 total),None,0
1,QA-02,True nulls,all columns,INFO,1 total true nulls across dataset — most missi...,None — handled in Check 3,1
2,QA-03,Sentinel -1,Rating,INFO,272 rows (12.1%) use -1 as missing encoding,Replaced with NaN,272
3,QA-03,Sentinel -1,Size,INFO,163 rows (7.2%) use -1 as missing encoding,Replaced with NaN,163
4,QA-03,Sentinel -1,Founded,WARNING,660 rows (29.3%) use -1 as missing encoding,Replaced with NaN,660
5,QA-03,Sentinel -1,Type of ownership,INFO,163 rows (7.2%) use -1 as missing encoding,Replaced with NaN,163
6,QA-03,Sentinel -1,Industry,INFO,353 rows (15.7%) use -1 as missing encoding,Replaced with NaN,353
7,QA-03,Sentinel -1,Sector,INFO,353 rows (15.7%) use -1 as missing encoding,Replaced with NaN,353
8,QA-03,Sentinel -1,Revenue,INFO,163 rows (7.2%) use -1 as missing encoding,Replaced with NaN,163
9,QA-03,Sentinel -1,Competitors,WARNING,1732 rows (76.9%) use -1 as missing encoding,Replaced with NaN,1732


---
## **Notes & Decisions**

| Decision | Rationale |
|---|---|
| All 2,252 rows retained | No rows removed beyond invalid titles — only values corrected |
| `-1` → `NaN`, not imputed | Imputation would introduce bias; NaN is more transparent |
| `Easy Apply` and `Competitors` excluded from clean file | 96% and 77% missing — not analytically useful |
| Seniority derived from title keywords | No seniority column in raw data; keyword method is consistent with bootcamp analysis |
| Founded outliers set to NaN | 26 implausible values more likely scraping errors than real data |
| Skill flags use regex on job description | Most reliable method without a structured skills field; patterns validated manually |

---
*This notebook was developed as part of a data analytics bootcamp sprint. The original cleaning was performed collaboratively in Python by DABC21 Group5. This notebook makes the process reproducible, documented, and auditable.*
